In [ ]:
!pip uninstall -y \
langgraph \
langgraph-prebuilt \
langgraph-checkpoint \
langgraph-sdk

!pip install -qU \
langchain==0.3.27 \
langchain-community==0.3.27 \
langchain-core==0.3.72 \
langchain-huggingface==0.3.1 \
langchain-text-splitters==0.3.9 \
transformers \
accelerate \
bitsandbytes \
sentence-transformers \
faiss-cpu \
pdfplumber \
pypdf \
datasets \
evaluate \
rouge_score \
ragas

import os
os.kill(os.getpid(), 9)

Found existing installation: langgraph 1.2.9
Uninstalling langgraph-1.2.9:
  Successfully uninstalled langgraph-1.2.9
Found existing installation: langgraph-prebuilt 1.1.0
Uninstalling langgraph-prebuilt-1.1.0:
  Successfully uninstalled langgraph-prebuilt-1.1.0
Found existing installation: langgraph-checkpoint 4.1.1
Uninstalling langgraph-checkpoint-4.1.1:
  Successfully uninstalled langgraph-checkpoint-4.1.1
Found existing installation: langgraph-sdk 0.4.2
Uninstalling langgraph-sdk-0.4.2:
  Successfully uninstalled langgraph-sdk-0.4.2
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 103.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.8/442.8 kB 41.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━

In [ ]:
import sys

print(sys.executable)

!pip list | grep langchain

/usr/bin/python3
langchain                             0.3.27
langchain-community                   0.3.27
langchain-core                        0.3.72
langchain-huggingface                 0.3.1
langchain-openai                      0.3.28
langchain-protocol                    0.0.18
langchain-text-splitters              0.3.9


In [ ]:
from langchain_community.llms import HuggingFacePipeline

print("Import berhasil")

Import berhasil


In [ ]:
from huggingface_hub import notebook_login
# Jalankan ini untuk memasukkan token Hugging Face Anda
# Anda bisa mendapatkan token di https://huggingface.co/settings/tokens
notebook_login()

In [ ]:
from google.colab import userdata
import os

try:
    # Mengambil token dari Colab Secrets
    os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
    print("✅ HF_TOKEN berhasil dimuat dari Secrets!")
except Exception as e:
    print("❌ Gagal memuat HF_TOKEN. Pastikan nama secret adalah 'HF_TOKEN' dan akses sudah diaktifkan.")

✅ HF_TOKEN berhasil dimuat dari Secrets!


In [ ]:
from google.colab import userdata
import os
from huggingface_hub import HfApi, whoami

try:
    # 1. Ambil token dari Secrets
    token = userdata.get('HF_TOKEN')
    os.environ["HF_TOKEN"] = token

    # 2. Verifikasi identitas user
    user_info = whoami(token=token)
    print(f"✅ Terhubung sebagai: {user_info['name']}")
    print(f"✅ Status Autentikasi: Sukses")

    # 3. Cek spesifik akses ke Llama 3
    model_id = "meta-llama/Meta-Llama-3-8B-Instruct"
    api = HfApi()
    try:
        api.model_info(model_id)
        print(f"✅ Akses ke {model_id}: TERSEDIA")
    except Exception as model_e:
        print(f"⚠️ Akses ke {model_id}: BELUM DISETUJUI (403 Forbidden)")
        print("Silakan klik 'Request Access' di: https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct")

except Exception as e:
    print(f"❌ Gagal Terhubung: {e}")
    print("Pastikan secret 'HF_TOKEN' sudah dibuat dan 'Notebook access' diaktifkan.")

✅ Terhubung sebagai: herdianset
✅ Status Autentikasi: Sukses
✅ Akses ke meta-llama/Meta-Llama-3-8B-Instruct: TERSEDIA


In [ ]:
import pdfplumber
import torch

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

pdf_path = "/content/drive/MyDrive/risetRAG/Pedoman_KP_BD.pdf"

def extract_text_from_pdf(pdf_path):
    text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text()
            if page_text:
                page_text = " ".join(page_text.split())
                text += page_text + " "
    return text

print("Membaca PDF...")
text_data = extract_text_from_pdf(pdf_path)
print("Jumlah karakter:", len(text_data))

# CHUNKING

splitter = RecursiveCharacterTextSplitter(
    chunk_size=512,
    chunk_overlap=50
)

chunks = splitter.split_text(text_data)

print("Jumlah chunk:", len(chunks))

# ====================================
# EMBEDDING
# ====================================

device = "cuda" if torch.cuda.is_available() else "cpu"

#embedding_model = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
embedding_model = "indobenchmark/indobert-base-p1"

embeddings = HuggingFaceEmbeddings(
    model_name=embedding_model,
    model_kwargs={
        "device": device
    },
    encode_kwargs={
        "normalize_embeddings": True
    }
)

print("Membuat FAISS...")

vectorstore = FAISS.from_texts(
    chunks,
    embeddings
)

# ====================================
# SAVE
# ====================================

save_path = "/content/drive/MyDrive/risetRAG/faiss_index"

vectorstore.save_local(save_path)

print("✅ FAISS berhasil disimpan")

Membaca PDF...
Jumlah karakter: 56383
Jumlah chunk: 122


config.json:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  498MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  498MB            

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/229k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Membuat FAISS...
✅ FAISS berhasil disimpan


In [ ]:
import torch

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

device = "cuda" if torch.cuda.is_available() else "cpu"

#embedding_model = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
embedding_model = "indobenchmark/indobert-base-p1"

embeddings = HuggingFaceEmbeddings(
    model_name=embedding_model,
    model_kwargs={
        "device": device
    },
    encode_kwargs={
        "normalize_embeddings": True
    }
)

print("Loading FAISS...")

vectorstore = FAISS.load_local(
    "/content/drive/MyDrive/risetRAG/faiss_index",
    embeddings,
    allow_dangerous_deserialization=True
)

print("✅ FAISS Loaded")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading FAISS...
✅ FAISS Loaded


In [ ]:
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    pipeline,
    BitsAndBytesConfig
)

from langchain_community.llms import HuggingFacePipeline

llama_model_id = "meta-llama/Meta-Llama-3-8B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("Loading Llama...")

tokenizer = AutoTokenizer.from_pretrained(
    llama_model_id
)

model = AutoModelForCausalLM.from_pretrained(
    llama_model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

# text_pipeline = pipeline(
#     "text-generation",
#     model=model,
#     tokenizer=tokenizer,
#     max_new_tokens=128,
#     temperature=0.1,
#     top_p=0.9,
#     do_sample=True
# )

text_pipeline = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=128,
    temperature=0,
    do_sample=False,
    return_full_text=False,
    pad_token_id=tokenizer.eos_token_id,
)

llm = HuggingFacePipeline(
    pipeline=text_pipeline
)

print("✅ Llama Loaded")

Loading Llama...


config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/51.0k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Passing `generation_config` together with generation-related arguments=({'temperature', 'do_sample', 'max_new_tokens', 'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


✅ Llama Loaded


/tmp/ipykernel_6349/3591529717.py:54: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(


In [ ]:
from langchain.chains.combine_documents import (
    create_stuff_documents_chain
)

from langchain.chains.retrieval import (
    create_retrieval_chain
)

from langchain_core.prompts import PromptTemplate

retriever = vectorstore.as_retriever(
    search_kwargs={
        "k": 3
    }
)

prompt = PromptTemplate.from_template(
"""
Anda adalah asisten akademik.
Gunakan konteks berikut untuk menjawab.
Jika informasi tidak ditemukan,
katakan bahwa informasi tidak ada
dalam dokumen.

Konteks:
{context}

Pertanyaan:
{input}

Jawaban:
"""
)

qa_chain = create_stuff_documents_chain(
    llm,
    prompt
)

rag_chain = create_retrieval_chain(
    retriever,
    qa_chain
)

print("✅ RAG Ready")

✅ RAG Ready


In [ ]:
chat_history = ""
print("=" * 60)
print("RAG CHAT AKADEMIK")
print("ketik exit untuk keluar")
print("=" * 60)

while True:
    question = input("\n Anda : ")
    if question.lower() in [
        "exit",
        "quit",
        "keluar"
    ]:
        break
    docs = retriever.invoke(question)

    context = "\n\n".join(
        [doc.page_content for doc in docs]
    )

    final_prompt = f"""
Riwayat Percakapan:
{chat_history}

Konteks:
{context}

Pertanyaan:
{question}

Jawaban:
"""
    answer = llm.invoke(final_prompt)
    print("\n Bot:")
    print(answer)
    chat_history += f"""
User: {question}
Assistant: {answer}
"""

🤖 RAG CHAT AKADEMIK
ketik exit untuk keluar

🧑 Anda : syarat kerja praktek


[transformers] Both `max_new_tokens` (=128) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.



🤖 Bot:

Riwayat Percakapan:


Konteks:
Kerja Praktek (KP) sebagaimana tercantum di dalam BAB III pedoman ini. 4 BAB III PROSEDUR KERJA PRAKTEK Berikut adalah prosedur Kerja Praktek (KP) yang berlaku bagi Mahasiswa yang akan mengajukan, melaksanakan, dan menyelesaikan Kerja Praktek (KP): Tahap 1 1. Mahasiswa yang akan melaksanakan Kerja Praktek (KP) wajib mengikuti sosialisi/pembekalan Kerja Praktek (KP) yang diadakan oleh Program Studi Bisnis Digital. 2. Mahasiswa yang akan melaksanakan Kerja Praktek (KP) dianjurkan untuk melakukan pendekatan

teori maksimal 10 tahun dari pembuatan laporan Kerja Praktek (KP). 4. HASIL KERJA PRAKTEK (TOPIK KERJA PRAKTEK) Judul pada bagian ini disesuaikan dengan topik Kerja Praktek (KP) yang diangkat untuk laporan. Lingkup/batasan pengerjaan laporan kerja praktek dapat disetarakan dengan minimal tahapan analisis/perencanaan dengan topik yang dapat diambil adalah sebagai berikut: a. Model Bisnis Topik ini bertujuan untuk mengeksplorasi fenomena rencana i

In [ ]:
#PENGUJIAN

import pandas as pd
from datasets import Dataset
from rouge_score import rouge_scorer

print("Menyiapkan Dataset Uji...")
test_data = [
    {
        "question": "Berapa SKS minimal telah ditempuh agar mahasiswa berhak mengajukan kerja praktek (KP)?",
        "ground_truth": "Mahasiswa telah menempuh minimal 110 SKS."
    },
    {
        "question": "Dimana tempat kerja praktek (KP) yang diijinkan?",
        "ground_truth": "Tempat Kerja Praktek (KP) yang diijinkan adalah instansi pemerintah, perusahaan  yang berbadan hukum minimal berbentuk CV"
    },
    {
        "question": "Siapa yang bisa melakukan penilaian atau evaluasi kerja praktek?",
        "ground_truth": "Pembina di Instansi/Perusahaan  dan Dosen Pembimbing di ITB STIKOM Bali."
    }
]

generated_results = []

print("="*60)
print("MEMULAI PENGUJIAN RAG")
print("="*60)

for item in test_data:

    query = item["question"]

    docs = retriever.invoke(query)

    context = "\n\n".join(
        [doc.page_content for doc in docs]
    )

    prompt = f"""
Anda adalah asisten akademik.

Jawablah pertanyaan HANYA berdasarkan konteks berikut.

Jika jawaban tidak ada pada konteks,
katakan bahwa informasi tidak ditemukan.

Konteks:
{context}

Pertanyaan:
{query}

Jawaban:
"""

    answer = llm.invoke(prompt).strip()

    generated_results.append({
        "user_input": query,
        "retrieved_contexts":[
            doc.page_content for doc in docs
        ],
        "response": answer,
        "reference": item["ground_truth"]
    })

print("Inferensi selesai.")

df = pd.DataFrame(generated_results)

display(df)

[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Menyiapkan Dataset Uji...
MEMULAI PENGUJIAN RAG


[transformers] Both `max_new_tokens` (=128) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_new_tokens` (=128) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `

Inferensi selesai.


,user_input,retrieved_contexts,response,reference
0,Berapa SKS minimal telah ditempuh agar mahasis...,[Praktek berakhir. 1 BAB II KETENTUAN UMUM KER...,110 SKS. (Informasi ditemukan pada konteks),Mahasiswa telah menempuh minimal 110 SKS.
1,Dimana tempat kerja praktek (KP) yang diijinkan?,"[perusahaan BUMN, BUMD, Software House, ISP (I...","Instansi pemerintah, institusi pendidikan, per...",Tempat Kerja Praktek (KP) yang diijinkan adala...
2,Siapa yang bisa melakukan penilaian atau evalu...,[bisa mengantarkan mahasiswa untuk dapat adapt...,Pembina di Instansi/Perusahaan dan Dosen Pembi...,Pembina di Instansi/Perusahaan dan Dosen Pemb...


In [ ]:
import numpy as np
from rouge_score import rouge_scorer

print("="*60)
print("EVALUASI ROUGE")
print("="*60)

scorer = rouge_scorer.RougeScorer(
    ["rouge1","rouge2","rougeL"],
    use_stemmer=False
)

rouge1=[]
rouge2=[]
rougeL=[]

for i,row in df.iterrows():

    score = scorer.score(
        row["reference"],
        row["response"]
    )

    rouge1.append(score["rouge1"].fmeasure)
    rouge2.append(score["rouge2"].fmeasure)
    rougeL.append(score["rougeL"].fmeasure)

    print("="*60)
    print(row["user_input"])
    print(f"ROUGE-1 : {score['rouge1'].fmeasure:.4f}")
    print(f"ROUGE-2 : {score['rouge2'].fmeasure:.4f}")
    print(f"ROUGE-L : {score['rougeL'].fmeasure:.4f}")

print("="*60)
print("RATA-RATA")
print("="*60)

print("ROUGE-1 :",np.mean(rouge1))
print("ROUGE-2 :",np.mean(rouge2))
print("ROUGE-L :",np.mean(rougeL))



EVALUASI ROUGE
Berapa SKS minimal telah ditempuh agar mahasiswa berhak mengajukan kerja praktek (KP)?
ROUGE-1 : 0.3333
ROUGE-2 : 0.2000
ROUGE-L : 0.3333
Dimana tempat kerja praktek (KP) yang diijinkan?
ROUGE-1 : 0.1159
ROUGE-2 : 0.0299
ROUGE-L : 0.1159
Siapa yang bisa melakukan penilaian atau evaluasi kerja praktek?
ROUGE-1 : 0.3333
ROUGE-2 : 0.3125
ROUGE-L : 0.3333
RATA-RATA
ROUGE-1 : 0.2608695652173913
ROUGE-2 : 0.18078358208955225
ROUGE-L : 0.2608695652173913


In [ ]:
# ==========================================================
# VALIDASI HALUSINASI (RAGAS)
# ==========================================================


import pandas as pd
import numpy as np
from datasets import Dataset
ragas_dataset = Dataset.from_pandas(df)

from ragas import evaluate
from ragas.metrics import faithfulness
from ragas.metrics import answer_relevancy
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

ragas_llm = LangchainLLMWrapper(llm)
ragas_embedding = LangchainEmbeddingsWrapper(embeddings)

result = evaluate(
    dataset=ragas_dataset,
    metrics=[
        faithfulness,
        answer_relevancy
    ],
    llm=ragas_llm,
    embeddings=ragas_embedding,
    raise_exceptions=True
)

ragas_df = result.to_pandas()

display(ragas_df)

print("=" * 60)
print("🛡️ EVALUASI HALUSINASI FAKTUAL (RAGAS)")
print("=" * 60)

# ----------------------------------------------------------
# 1. Konversi Dataset
# ----------------------------------------------------------
ragas_dataset = Dataset.from_pandas(pd.DataFrame(generated_results))

# ----------------------------------------------------------
# 2. Wrapper LLM & Embedding
# ----------------------------------------------------------
ragas_llm = LangchainLLMWrapper(llm)
ragas_emb = LangchainEmbeddingsWrapper(embeddings)

# ----------------------------------------------------------
# 3. Evaluasi
# ----------------------------------------------------------
result = evaluate(
    dataset=ragas_dataset,
    metrics=[
        faithfulness,
        answer_relevancy
    ],
    llm=ragas_llm,
    embeddings=ragas_emb
)

# ----------------------------------------------------------
# 4. Konversi ke DataFrame
# ----------------------------------------------------------
ragas_df = result.to_pandas()

print("\nDetail Hasil Evaluasi")
print(ragas_df.head())

# ----------------------------------------------------------
# 5. Hitung Nilai Rata-rata
# ----------------------------------------------------------
print("\n" + "=" * 60)
print("RATA-RATA SKOR RAGAS")
print("=" * 60)

if "faithfulness" in ragas_df.columns:
    faithfulness_score = ragas_df["faithfulness"].mean()
    print(f"✅ Faithfulness       : {faithfulness_score:.4f}")
else:
    print("❌ Kolom 'faithfulness' tidak ditemukan.")

if "answer_relevancy" in ragas_df.columns:
    relevancy_score = ragas_df["answer_relevancy"].mean()
    print(f"✅ Answer Relevancy   : {relevancy_score:.4f}")
else:
    print("❌ Kolom 'answer_relevancy' tidak ditemukan.")

# ----------------------------------------------------------
# 6. Statistik Tambahan
# ----------------------------------------------------------
print("\n" + "=" * 60)
print("STATISTIK DETAIL")
print("=" * 60)

metric_columns = [
    col for col in [
        "faithfulness",
        "answer_relevancy"
    ] if col in ragas_df.columns
]

if len(metric_columns) > 0:
    print(ragas_df[metric_columns].describe())
else:
    print("Tidak ada metric yang dapat ditampilkan.")

# ----------------------------------------------------------
# 7. Simpan ke CSV
# ----------------------------------------------------------
output_file = "hasil_evaluasi_ragas.csv"
ragas_df.to_csv(output_file, index=False)

print("\n📁 Detail evaluasi berhasil disimpan ke:")
print(output_file)

print("\n✅ Evaluasi RAGAS selesai.")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)
/tmp/ipykernel_6349/2107020624.py:12: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness
/tmp/ipykernel_6349/2107020624.py:13: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.m

Evaluating:   0%|          | 0/6 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=128) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hugging

TimeoutError: 

In [ ]:
#RAGAS BARU

ragas_dataset=Dataset.from_pandas(df)
result=evaluate(

    dataset=ragas_dataset,

    metrics=[
        faithfulness,
        answer_relevancy
    ],

    llm=ragas_llm,

    embeddings=ragas_embedding,

    raise_exceptions=True

)

Evaluating:   0%|          | 0/6 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=128) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hugging

TimeoutError: 